## Overview

This notebook is intended to:

* Display the yearly net flow per tariff group. 

This snippet notebook is designed as follows:

* Define the grid and year of interest.
* Sum the net flow per tariff group.
* Present and plot the results. 

This snippet presents two methods of querying, aggregating and calculating these results. The first method uses a single SQL query to fetch the data, aggregate, and calculate the results. The second method uses a SQL query to fetch the data and Python to aggregate it and calculate the results.

---

## Setup

In [1]:
import getpass
import urllib.parse
import plotly.express as px
import pandas as pd
import numpy as np
import datetime
import pytz

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

%load_ext sql
%sql postgresql://$edm_name:$edm_password@$edm_address/edm
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False

# Delete the credential variables for security purposes.
del edm_name, edm_password


EDM login information


---

## Net Flow per Tariff Group

#### Input Parameters
Enter the grid ID and year. 

In [3]:
grid_id = input('Enter grid ID: ') # awefice
year = input('Enter year: ') # 2021

# Get the start and end dates for the year and localize them.
year_start = datetime.datetime.min.replace(year = int(year))
year_start = pytz.timezone('America/Vancouver').localize(year_start)
year_end = datetime.datetime.max.replace(year = int(year))
year_end = pytz.timezone('America/Vancouver').localize(year_end)

# Convert it to string. 
timerange_tz = f'[ {year_start} , {year_end} ]'

#### Sum of net flow by tariff group - Method 1 - Using only SQL to fetch, aggregate and calculate results

In [4]:
%%sql net_tariff <<

SELECT ge.meta ->> 'tariff_id' as tariff_id, 
    ROUND(CAST(SUM(tdss_c.value - COALESCE(tdss_p.value,0)) AS NUMERIC),2) as "net_kWh"
FROM grid_element ge 
JOIN grid_element_data_source geds_c 
    ON geds_c.grid_element_id = ge.grid_element_id 
    AND geds_c.type = 'CONSUMER' 
JOIN ts_data_source_select(geds_c.grid_element_data_source_id, 'kWh', '{timerange_tz}') tdss_c 
    ON true 
LEFT JOIN grid_element_data_source geds_p 
    ON geds_p.grid_element_id = geds_c.grid_element_id 
    AND geds_p.type = 'PRODUCER' 
LEFT JOIN ts_data_source_select(geds_p.grid_element_data_source_id, 'kWh', '{timerange_tz}') tdss_p 
    ON tdss_p.timestamp = tdss_c.timestamp 
WHERE ge.grid_id = '{grid_id}'
    AND ge.type = 'Meter'
GROUP BY ge.meta ->> 'tariff_id'
ORDER BY ge.meta ->> 'tariff_id';

Returning data to local variable net_tariff


In [5]:
# Convert the results to a data frame.
df_tariff = net_tariff.DataFrame()
df_tariff['net_kWh'] = df_tariff['net_kWh'].astype(float)
df_tariff

,tariff_id,net_kWh
0,comm_basic,29974.02
1,comm_ev,102277.42
2,comm_green,101043.92
3,res_basic,192903.04
4,res_ev,29006.09
5,res_green,23689.15


In [6]:
fig = px.bar(df_tariff, x="net_kWh",  y="tariff_id", orientation="h", color="net_kWh",
             color_continuous_scale=px.colors.diverging.Temps, color_continuous_midpoint=0,
             title="<b>Net Flow by Tariff Group for the year {}<b>".format(year),
             labels={'tariff_id':'Tariff Group', 
                         'net_kWh': 'Net Flow (kWh)'})
fig.show()

Negative net flow indicates that production from DERs (in this case, solar panels) is greater than the current consumption. 

---

#### Sum of net flow by tariff group - Method 2 - Using a SQL query to fetch the data and Python to aggregate and calculate the results  

In [7]:
%%sql net_flow <<

SELECT ge.meta ->> 'tariff_id' as tariff_id, 
    tdss.value as "kWh",
    geds.type
FROM grid_element ge 
JOIN grid_element_data_source geds 
    ON geds.grid_element_id = ge.grid_element_id  
JOIN ts_data_source_select(geds.grid_element_data_source_id, 'kWh', '{timerange_tz}') tdss 
    ON true 
WHERE ge.grid_id = '{grid_id}'
    AND ge.type = 'Meter';

Returning data to local variable net_flow


In [8]:
# Convert the results to a data frame.
df_flow = net_flow.DataFrame()

In [9]:
# Aggregate the consumer load.
df_consumer = df_flow.loc[df_flow['type']=='CONSUMER'].groupby('tariff_id').sum().rename(columns={'kWh': 'consumer_kWh'})

# Aggregate the producer load.
df_producer = df_flow.loc[df_flow['type']=='PRODUCER'].groupby('tariff_id').sum().rename(columns={'kWh': 'producer_kWh'})

# Concat the aggregated consumer and producer loads. 
df_net=pd.concat([df_consumer, df_producer], axis=1).fillna(0).reset_index()

# Calculate the net load. 
df_net['net_kWh']=(df_net['consumer_kWh']-df_net['producer_kWh']).round(2)

# Display the results. 
df_net.sort_values(by=['tariff_id'])

,tariff_id,consumer_kWh,type,producer_kWh,type,net_kWh
0,comm_basic,29974.022000,CONSUMERCONSUMERCONSUMERCONSUMERCONSUMERCONSUM...,0.000000,0,29974.02
1,comm_ev,102277.423422,CONSUMERCONSUMERCONSUMERCONSUMERCONSUMERCONSUM...,0.000000,0,102277.42
2,comm_green,160860.940259,CONSUMERCONSUMERCONSUMERCONSUMERCONSUMERCONSUM...,59817.023492,PRODUCERPRODUCERPRODUCERPRODUCERPRODUCERPRODUC...,101043.92
3,res_basic,192903.044378,CONSUMERCONSUMERCONSUMERCONSUMERCONSUMERCONSUM...,0.000000,0,192903.04
4,res_ev,29006.088453,CONSUMERCONSUMERCONSUMERCONSUMERCONSUMERCONSUM...,0.000000,0,29006.09
5,res_green,27242.504566,CONSUMERCONSUMERCONSUMERCONSUMERCONSUMERCONSUM...,3553.357047,PRODUCERPRODUCERPRODUCERPRODUCERPRODUCERPRODUC...,23689.15


---